# 04 - Full Dataset Mass Inference

Implements **Step 5** of `classification_strategy.md`. Applies the trained Twitter-RoBERTa
to every tweet in the **leftover corpus** — the tweets that were NOT in the partitioned
set used for HITL training (~17M tweets per the project README).

Reuses the labels produced by `03_final_inference.ipynb` so we never re-classify a tweet
we already have a label for, and applies the same retweet split + orphan-promotion
pattern as 03 to keep retweets consistent with their referenced originals.

Designed for Colab GPU runs. Saves checkpoints every `SAVE_EVERY` tweets so a Colab
disconnect loses at most that many predictions.

In [ ]:
%%time
import os
from pathlib import Path
import sys

# --- ENVIRONMENT SWITCH ---
# True  → local machine with Google Drive Desktop mounted
# False → Google Colab cloud
RUNNING_LOCALLY = False

# --- DATASET TYPE ---
# 'AI'  → AItrust_twits_pruned_dict.json    → Partitioned Data/AI Data/
# 'Art' → AItrust_Art_pruned_twit_dict.json → Partitioned Data/Art Data/
DATASET_TYPE = 'AI'

if RUNNING_LOCALLY:
    _REPO_ROOT = str(Path(os.getcwd()).resolve().parents[1])
    if _REPO_ROOT not in sys.path:
        sys.path.insert(0, _REPO_ROOT)
    BASE_PATH = Path('/Volumes/GoogleDrive/My Drive/Colab Projects/AI Public Trust')
else:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE_PATH = Path('/content/drive/My Drive/Colab Projects/AI Public Trust')

# Pre-compute critical paths
twits_folder          = BASE_PATH / 'Raw Data/Twits/'
test_folder           = BASE_PATH / 'Raw Data/'
datasets_folder       = BASE_PATH / 'Data Sets'
cleanedds_folder      = BASE_PATH / 'Data Sets/Cleaned Data'
networks_folder       = BASE_PATH / 'Data Sets/Networks/'
literature_folder     = BASE_PATH / 'Literature/'
topic_models_folder   = BASE_PATH / 'Models/Topic Modeling/'
classifiers_folder    = BASE_PATH / 'Models/Classifiers/'
classifiers_folder.mkdir(parents=True, exist_ok=True)
partitioned_folder    = cleanedds_folder / 'Partitioned Data' / f'{DATASET_TYPE} Data'
hitl_folder           = datasets_folder / 'Classifiers_Data' / 'HITL'
final_folder          = datasets_folder / 'Classifiers_Data' / 'Final'
full_inference_folder = datasets_folder / 'Classifiers_Data' / 'Full_Inference'
full_inference_folder.mkdir(parents=True, exist_ok=True)

In [ ]:
%%time
import os
if not RUNNING_LOCALLY:
    print('Running Colab setup...')
    import subprocess
    subprocess.run(['pip', 'install', '-q', 'transformers', 'torch', 'tqdm'])
else:
    print('Running locally: skipping Colab setup.')

In [ ]:
%%time
import pickle
import time
import numpy as np
import pandas as pd
import torch
import tqdm
from transformers import AutoTokenizer, AutoModelForSequenceClassification, pipeline

# Configuration

**`FULL_CORPUS_PATH` is a TODO** — the project README references a ~17M tweet corpus
but does not pin a single canonical file path. Set this to whatever JSONL/PKL holds
the full corpus (`AItrust_twits_dict.json` is the unprocessed dump from
`02_Processing/01`; if you have a larger file from elsewhere, point at that). The
loader handles `.json` (one tweet per line), `.jsonl`, `.jsonl.gz`, and `.pkl`.

In [ ]:
%%time
# ── Inference batching ─────────────────────────────────────────────────
ROBERTA_BATCH = 64
ROBERTA_MULTI = 8                          # outer chunk = ROBERTA_BATCH * ROBERTA_MULTI
OUTER_BATCH   = ROBERTA_BATCH * ROBERTA_MULTI
CHUNK_SIZE    = 50_000                     # rows loaded into memory per Pass 1 chunk
SAVE_EVERY    = 100_000                    # checkpoint cadence

# ── Inputs ─────────────────────────────────────────────────────────────
# TODO: confirm the path to the full corpus.
_FULL_CORPUS_PATHS = {
    'AI':  datasets_folder / 'AItrust_twits_dict.json',
    'Art': datasets_folder / 'AItrust_Art_twit_dict.json',  # TODO: confirm Art corpus path
}
FULL_CORPUS_PATH = _FULL_CORPUS_PATHS[DATASET_TYPE]

# Final-inference output from notebook 03 (the labelled partitioned corpus).
FINAL_LABELLED_PATH = final_folder / 'final_annotated_tweets.pkl'
PARTITION_MANIFEST  = partitioned_folder / 'partition_ids.pkl'

# 1. Load the trained model

In [ ]:
%%time
best_path = classifiers_folder / 'best_roberta_model'
if not best_path.exists():
    raise FileNotFoundError(f'Model not found at {best_path}. Run notebook 02 first.')

tokenizer = AutoTokenizer.from_pretrained(str(best_path))
model     = AutoModelForSequenceClassification.from_pretrained(str(best_path))
device    = 0 if torch.cuda.is_available() else -1
clf_pipe  = pipeline(
    'text-classification', model=model, tokenizer=tokenizer,
    device=device, batch_size=ROBERTA_BATCH, return_all_scores=True,
    truncation=True, max_length=128,
)
print(f'Model loaded. Device: {"GPU" if device == 0 else "CPU"}')

def predict_top(texts):
    """Run clf_pipe over an iterable of texts and return [(label, confidence), ...]."""
    out = []
    for i in range(0, len(texts), OUTER_BATCH):
        for scores in clf_pipe(texts[i:i + OUTER_BATCH]):
            top = max(scores, key=lambda x: x['score'])
            out.append((top['label'], float(top['score'])))
    return out

# 2. Load existing labels from notebook 03

We seed the `labels` dict from `final_annotated_tweets.pkl` so that any tweet that's
already in the partitioned/HITL corpus is skipped during Pass 1 here (no double work,
and the human-confirmed labels stay authoritative). Provenance is preserved.

In [ ]:
%%time
if not FINAL_LABELLED_PATH.exists():
    raise FileNotFoundError(
        f'{FINAL_LABELLED_PATH} not found. Run 03_final_inference.ipynb first '
        f'so notebook 04 has its labels seed.'
    )

seed = pd.read_pickle(FINAL_LABELLED_PATH)
seed['id'] = seed['id'].astype(str)
labels: dict[str, dict] = {}
for row in seed[['id', 'predicted_label', 'confidence', 'label_source']].itertuples(index=False):
    labels[row.id] = {
        'label':        row.predicted_label,
        'confidence':   float(row.confidence) if pd.notna(row.confidence) else 1.0,
        'label_source': row.label_source if pd.notna(row.label_source) else 'unknown',
    }
print(f'Seeded labels dict from notebook 03: {len(labels):,}')

# 3. Load the full corpus and filter to new tweets

Drops every tweet that's already in `partition_ids.pkl` (i.e., already labelled by 03)
and normalises `id`, `text`, `processed_text` (when present), `type`, `likes`,
`retweets`, and `referenced_tweets_dictionary` into a single dataframe.

In [ ]:
%%time
def _load_corpus(path):
    s = path.suffix.lower()
    if s == '.pkl':
        return pd.read_pickle(path)
    if s == '.json':
        return pd.read_json(path, lines=True)
    if s == '.jsonl':
        return pd.read_json(path, lines=True)
    if path.name.endswith('.jsonl.gz') or path.name.endswith('.json.gz'):
        return pd.read_json(path, lines=True, compression='gzip')
    if s == '.csv':
        return pd.read_csv(path)
    raise ValueError(f'Unsupported corpus format: {path}')

if not FULL_CORPUS_PATH.exists():
    raise FileNotFoundError(
        f'{FULL_CORPUS_PATH} not found. Set FULL_CORPUS_PATH in the Configuration cell.'
    )
print(f'Loading {FULL_CORPUS_PATH}...')
t0 = time.time()
full_df = _load_corpus(FULL_CORPUS_PATH)
print(f'Loaded {len(full_df):,} rows in {time.time() - t0:.1f}s')
print(f'Columns: {list(full_df.columns)}')

# Normalise the columns we care about; tolerate missing optional fields.
full_df['id'] = full_df['id'].astype(str)
full_df['text'] = full_df.get('text', pd.Series([''] * len(full_df))).astype(str)
if 'processed_text' not in full_df.columns:
    full_df['processed_text'] = ''
if 'type' not in full_df.columns:
    full_df['type'] = 'unknown'

def _pm_field(pm, key, default=0):
    return pm.get(key, default) if isinstance(pm, dict) else default

if 'public_metrics' in full_df.columns:
    full_df['likes']    = full_df['public_metrics'].apply(
        lambda pm: _pm_field(pm, 'like_count', 0)).astype(int)
    full_df['retweets'] = full_df['public_metrics'].apply(
        lambda pm: _pm_field(pm, 'retweet_count', 0)).astype(int)
elif 'likes' not in full_df.columns:
    full_df['likes'] = 0
    full_df['retweets'] = 0

# Drop tweets we already labelled in 03 (much faster as a set membership check).
if PARTITION_MANIFEST.exists():
    with open(PARTITION_MANIFEST, 'rb') as f:
        partition_ids = pickle.load(f)
    already_labelled = set()
    for ids in partition_ids.values():
        already_labelled.update(map(str, ids))
    print(f'Already labelled in 03: {len(already_labelled):,} ids')
else:
    already_labelled = set(labels.keys())
    print(f'Manifest missing — falling back to labels-dict membership ({len(already_labelled):,})')

new_df = full_df[~full_df['id'].isin(already_labelled)].reset_index(drop=True)
print(f'New tweets to classify: {len(new_df):,}')

# Split retweets out (same protocol as 00).
RETWEET_TYPE = 'retweeted'
new_retweets = new_df[new_df['type'] == RETWEET_TYPE].reset_index(drop=True)
new_originals = new_df[new_df['type'] != RETWEET_TYPE].reset_index(drop=True)
print(f'  new originals/replies/quotes: {len(new_originals):,}')
print(f'  new retweets:                 {len(new_retweets):,}')

# 4. Pass 1 — Classify new non-retweets, with checkpointing

Runs RoBERTa over `new_originals` in chunks of `CHUNK_SIZE`. Every `SAVE_EVERY` rows the
intermediate `labels` dict is pickled to `Full_Inference/checkpoint_<n>.pkl` so a Colab
disconnect loses at most `SAVE_EVERY` predictions.

In [ ]:
%%time
def _checkpoint(n_processed):
    cp = full_inference_folder / f'checkpoint_{n_processed}.pkl'
    with open(cp, 'wb') as f:
        pickle.dump(labels, f)
    print(f'  checkpoint → {cp.name} (labels dict size: {len(labels):,})')

n_total = len(new_originals)
if n_total:
    t_start = time.time()
    last_saved = 0
    for chunk_start in tqdm.tqdm(range(0, n_total, CHUNK_SIZE),
                                  total=(n_total + CHUNK_SIZE - 1) // CHUNK_SIZE,
                                  desc='Pass 1'):
        chunk = new_originals.iloc[chunk_start:chunk_start + CHUNK_SIZE]
        preds = predict_top(chunk['text'].astype(str).tolist())
        for tid, (label, conf) in zip(chunk['id'].tolist(), preds):
            labels[tid] = {
                'label':        label,
                'confidence':   conf,
                'label_source': 'model_original',
            }
        processed = chunk_start + len(chunk)
        if processed - last_saved >= SAVE_EVERY:
            _checkpoint(processed)
            last_saved = processed
    if last_saved != n_total:
        _checkpoint(n_total)
    print(f'Pass 1 total: {time.time() - t_start:.1f}s on {n_total:,} new originals')
else:
    print('No new non-retweets to classify.')

# 5. Pass 2 — Promote orphan originals among new retweets

Same logic as notebook 03's Pass 2, but operating on `new_retweets`.

In [ ]:
%%time
def _ref_id(rd):
    if isinstance(rd, dict):
        rid = rd.get('id')
        if rid is not None:
            return str(rid)
    return None

# Prefer the precomputed `ref_id` column where available (e.g. corpus produced
# by an updated 02_Processing step). Fall back to extracting from the nested dict.
if 'ref_id' in new_retweets.columns:
    new_retweets['ref_id'] = new_retweets['ref_id'].where(new_retweets['ref_id'].notna(), None)
elif 'referenced_tweets_dictionary' in new_retweets.columns:
    new_retweets['ref_id'] = new_retweets['referenced_tweets_dictionary'].apply(_ref_id)
else:
    print('WARNING: full corpus has no ref_id / referenced_tweets_dictionary column.')
    print('         Every retweet will fall through to the no-reference fallback in Pass 3.')
    new_retweets['ref_id'] = None

n_no_ref = new_retweets['ref_id'].isna().sum()
rate_no_ref = n_no_ref / max(len(new_retweets), 1)
print(f'Retweets with no usable ref_id: {n_no_ref:,} ({rate_no_ref:.1%})')

ref_in_labels = new_retweets['ref_id'].isin(labels)
orphan_mask   = new_retweets['ref_id'].notna() & ~ref_in_labels
orphans       = new_retweets[orphan_mask]
print(f'Standard-lookup retweets: {ref_in_labels.sum():,}')
print(f'Orphan retweets:          {len(orphans):,}')

orphan_originals: dict[str, dict] = {}
if len(orphans):
    sorted_orphans = (orphans
                      .assign(_eng=lambda d: d['likes'] + d['retweets'])
                      .sort_values(['_eng', 'id'], ascending=[False, True]))
    representatives = sorted_orphans.drop_duplicates(subset='ref_id', keep='first')
    print(f'Distinct missing originals (model calls): {len(representatives):,}')

    t0 = time.time()
    rep_preds = predict_top(representatives['text'].astype(str).tolist())
    print(f'Pass 2 RoBERTa: {time.time() - t0:.1f}s')

    for ref_id, (label, conf) in zip(representatives['ref_id'].tolist(), rep_preds):
        orphan_originals[ref_id] = {
            'label':        label,
            'confidence':   conf,
            'label_source': 'model_synthetic_retweet',
        }
print(f'orphan_originals dict size: {len(orphan_originals):,}')

shared = set(labels.keys()) & set(orphan_originals.keys())
assert not shared, f'Invariant violated: {len(shared)} ref_ids in both dicts'

# 6. Pass 3 — Look up retweet labels

In [ ]:
%%time
retweet_records = []
no_ref_rows    = []

for row in new_retweets.itertuples(index=False):
    ref_id = row.ref_id
    if ref_id in labels:
        rec = labels[ref_id]
    elif ref_id in orphan_originals:
        rec = orphan_originals[ref_id]
    else:
        no_ref_rows.append((row.id, str(row.text)))
        continue
    retweet_records.append({
        'id':              row.id,
        'predicted_label': rec['label'],
        'confidence':      rec['confidence'],
        'label_source':    'lookup',
    })

if no_ref_rows:
    print(f'No-reference retweets to classify directly: {len(no_ref_rows):,}')
    t0 = time.time()
    no_ref_preds = predict_top([t for _, t in no_ref_rows])
    print(f'Pass 3 fallback RoBERTa: {time.time() - t0:.1f}s')
    for (tid, _t), (label, conf) in zip(no_ref_rows, no_ref_preds):
        retweet_records.append({
            'id':              tid,
            'predicted_label': label,
            'confidence':      conf,
            'label_source':    'model_no_reference',
        })

retweet_label_df = pd.DataFrame(retweet_records).set_index('id')
print(f'New retweet labels assembled: {len(retweet_label_df):,}')

# 7. Assemble + save the full mass-inference output

Combines the seed (notebook 03's output) with the new originals classified in Pass 1
and the new retweets resolved in Pass 3 into one annotated table.

In [ ]:
%%time
# 1. The seed is already a fully assembled dataframe — keep it as the base.
seed_indexed = seed.drop_duplicates(subset='id', keep='first').set_index('id')

# 2. New originals: join metadata with their freshly written labels.
new_orig_label_df = pd.DataFrame.from_dict(
    {tid: labels[tid] for tid in new_originals['id'] if tid in labels},
    orient='index',
).rename(columns={'label': 'predicted_label'})
new_orig_label_df.index.name = 'id'
new_orig_meta = new_originals.set_index('id')
new_orig_final = new_orig_meta.join(new_orig_label_df, how='left')

# 3. New retweets: join metadata with the lookup-derived labels.
drop_cols = [c for c in ('predicted_label', 'human_label', 'ref_id')
             if c in new_retweets.columns]
new_rt_meta = new_retweets.drop(columns=drop_cols).set_index('id')
new_rt_final = new_rt_meta.join(retweet_label_df, how='left')

# 4. Concatenate — seed first so its provenance wins on any unexpected id collision.
full_annotated = pd.concat([seed_indexed, new_orig_final, new_rt_final], axis=0)
full_annotated = full_annotated[~full_annotated.index.duplicated(keep='first')].reset_index()

preferred = ['id', 'text', 'processed_text', 'type', 'likes', 'retweets',
             'predicted_label', 'confidence', 'label_source']
cols = [c for c in preferred if c in full_annotated.columns] + \
       [c for c in full_annotated.columns if c not in preferred]
full_annotated = full_annotated[cols]

print(f'Full annotated corpus: {len(full_annotated):,} rows')
print()
print('label_source distribution:')
print(full_annotated['label_source'].value_counts(dropna=False).to_string())
print()
print('predicted_label distribution (top 20):')
print(full_annotated['predicted_label'].value_counts(dropna=False).head(20).to_string())

out_pkl = full_inference_folder / 'full_inference_annotated.pkl'
out_csv = full_inference_folder / 'full_inference_annotated.csv'
full_annotated.to_pickle(out_pkl)
full_annotated.to_csv(out_csv, index=False)
print(f'\nSaved → {out_pkl}')
print(f'Saved → {out_csv}')

In [ ]:
# Disconnect from Colab runtime (no-op locally)
try:
    from google.colab import runtime
    runtime.unassign()
except ImportError:
    pass
